# Statistics for Machine Learning
The statistics every machine learning project uses, explained simply. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The centre: mean, median, mode
Three ways to describe a *typical* value. The **mean** is the average, the **median** is the middle value, the **mode** is the most common value.

### The mean, by hand and with NumPy
- `times`: seven delivery times, in minutes
- `times.sum(), len(times)`: the total (224) and how many there are (7)
- `sum / len`: mean = total divided by how many: 224 / 7 = 32
- `np.mean(times)`: NumPy does the same in one call

In [ ]:
import numpy as np

times = np.array([22, 25, 25, 28, 30, 31, 63])
print(times.sum(), len(times))
print(times.sum() / len(times))
print(np.mean(times))

### The median, and the effect of one extreme value
- `np.median`: sort the values and take the middle one: 28
- `times[times < 60]`: a mask that removes the 63-minute delivery
- `mean vs median`: removing one value moves the mean by 5 minutes, but the median by only 1.5

In [ ]:
print(np.median(times))

times_no_outlier = times[times < 60]
print(np.mean(times_no_outlier))
print(np.median(times_no_outlier))

### The mode
- `from statistics import mode`: Python's built-in statistics module has a mode function
- `mode(times)`: the most common value: 25 appears twice

In [ ]:
from statistics import mode
print(mode(times))

## 2. The spread: range, variance, standard deviation
Two datasets can share a mean but be very different. **Spread** measures how far values are from the centre.

### Range
- `max - min`: the range: biggest minus smallest (63 - 22 = 41). Simple, but one extreme value controls it

In [ ]:
print(times.max() - times.min())

### Deviations and squares
- `times - times.mean()`: how far each value is from the mean (negative = below, positive = above)
- `deviations ** 2`: square them: no more negatives, and big distances count much more
- `squares.sum()`: add them up: 1180

In [ ]:
deviations = times - times.mean()
print(deviations)
squares = deviations ** 2
print(squares)
print(squares.sum())

### Variance and standard deviation
- `squares.sum() / len(times)`: variance = the average squared distance from the mean: 1180 / 7
- `np.var`: NumPy's variance gives the same answer
- `np.sqrt(variance)`: standard deviation = square root of the variance, back in minutes: about 13
- `np.std`: NumPy's standard deviation

In [ ]:
variance = squares.sum() / len(times)
print(variance)
print(np.var(times))
std = np.sqrt(variance)
print(std)
print(np.std(times))

### Sample vs population: dividing by n - 1
- `ddof=1`: divide by n - 1 instead of n; used when the data is a sample of a bigger group
- `pandas .std()`: pandas divides by n - 1 by default, NumPy by n: that's why their answers can differ

In [ ]:
print(np.std(times, ddof=1))   # sample: divide by n - 1

import pandas as pd
print(pd.Series(times).std())   # pandas uses n - 1 by default

## 3. A real dataset: 500 deliveries
A pizza shop recorded 500 deliveries: the distance in km and the delivery time in minutes.

### Load the data
- `url = (...)`: the web address of the data file, split over three lines
- `pd.read_csv`: load it into a DataFrame
- `df.shape`: 500 rows, 3 columns

In [ ]:
url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "statistics-basics/deliveries.csv")
df = pd.read_csv(url)
print(df.shape)
df.head()

### Summary
- `describe()`: count, mean, std, min, quartiles and max in one go
- `.round(1)`: one decimal place

In [ ]:
df["time_min"].describe().round(1)

### Percentiles and quartiles
- `np.percentile(t, 90)`: 90% of deliveries take this long or less
- `q1, q3`: the 25th and 75th percentiles: the quartiles
- `iqr = q3 - q1`: interquartile range: the spread of the middle half

In [ ]:
t = df["time_min"]
print(np.percentile(t, 90))
q1, q3 = np.percentile(t, [25, 75])
iqr = q3 - q1
print(q1, q3, iqr)

### Finding outliers with the IQR rule
- `q3 + 1.5 * iqr`: the usual fence: anything above it is an outlier
- `|`: or: above the upper fence OR below the lower
- `sort_values().tolist()`: the outlier times, smallest first, as a list

In [ ]:
upper = q3 + 1.5 * iqr
lower = q1 - 1.5 * iqr
print(lower, upper)
outliers = df[(t > upper) | (t < lower)]
print(len(outliers))
print(outliers["time_min"].sort_values().tolist())

### Box plot
- `plt.boxplot(t)`: box = middle half (Q1 to Q3), line = median, whiskers = up to the fences, dots = outliers

In [ ]:
import matplotlib.pyplot as plt

plt.boxplot(t)
plt.title("Delivery times")
plt.ylabel("minutes")
plt.show()

## 4. Distributions
A **distribution** is the shape of the data: which values are common and which are rare. A histogram shows it.

### Delivery times: skewed right
- `plt.axvline`: a vertical line: red for the mean, dashed green for the median
- `the long right tail`: a few slow deliveries pull the mean above the median: the data is skewed right

In [ ]:
plt.hist(t, bins=25, edgecolor="black")
plt.axvline(t.mean(), color="red", label="mean")
plt.axvline(t.median(), color="green", linestyle="--", label="median")
plt.legend()
plt.title("Delivery times")
plt.xlabel("minutes")
plt.ylabel("orders")
plt.show()

### The normal distribution
- `rng.normal(170, 8, 10000)`: 10,000 random heights from a normal distribution with mean 170 and standard deviation 8
- `bell shape`: symmetric, most values near the middle, fewer and fewer towards the edges

In [ ]:
rng = np.random.default_rng(0)
heights = rng.normal(170, 8, 10000)   # mean 170 cm, std 8 cm
plt.hist(heights, bins=40, edgecolor="black")
plt.title("Heights: a normal distribution")
plt.xlabel("height (cm)")
plt.show()

### The 68-95-99.7 rule
- `np.abs(heights - m) <= k * s`: True for heights within k standard deviations of the mean
- `np.mean(True/False)`: the share of True values
- `result`: about 68%, 95% and 99.7%

In [ ]:
m, s = heights.mean(), heights.std()
for k in [1, 2, 3]:
    share = np.mean(np.abs(heights - m) <= k * s)
    print(k, "std:", round(share * 100, 1), "%")

### z-scores
- `(190 - m) / s`: z-score: how many standard deviations a value is from the mean
- `z_times`: a z-score for every delivery; the slowest is more than 4 standard deviations above average

In [ ]:
z = (190 - m) / s
print(round(z, 2))
z_times = (t - t.mean()) / t.std()
print(round(z_times.max(), 2))

## 5. Correlation
**Correlation** measures how strongly two things move together, from -1 to +1.

### Distance vs time
- `alpha=0.4`: see-through dots, so overlapping points show up darker

In [ ]:
plt.scatter(df["distance_km"], t, alpha=0.4)
plt.title("Distance vs delivery time")
plt.xlabel("distance (km)")
plt.ylabel("minutes")
plt.show()

### The correlation coefficient
- `np.corrcoef(x, y)[0, 1]`: the correlation between distance and time: about 0.78
- `.corr()`: pandas makes a table of correlations between every pair of columns

In [ ]:
print(np.corrcoef(df["distance_km"], t)[0, 1].round(2))
print(df[["distance_km", "time_min"]].corr().round(2))

## 6. Mini project: is a 40-minute promise realistic?
The shop wants to promise delivery within 40 minutes. What does the data say?

### How often would we keep the promise?
- `(t <= promise).mean()`: the share of deliveries that made it in 40 minutes
- `percentile 90 / 95`: the promise you could keep 90% or 95% of the time

In [ ]:
promise = 40
on_time = (t <= promise).mean()
print("On time:", round(on_time * 100, 1), "%")
print("90% arrive within", np.percentile(t, 90), "minutes")
print("95% arrive within", np.percentile(t, 95), "minutes")

### Near vs far orders
- `t[df["distance_km"] > 8]`: delivery times for orders more than 8 km away
- `compare`: the promise works for near orders but fails for far ones

In [ ]:
far = t[df["distance_km"] > 8]
near = t[df["distance_km"] <= 8]
print("Far orders on time:", round((far <= promise).mean() * 100, 1), "%")
print("Near orders on time:", round((near <= promise).mean() * 100, 1), "%")

## Try it yourself
1. Remove the 63-minute delivery from `times` and recalculate the mean, median and standard deviation.
2. What delivery promise would be met 95% of the time?
3. Draw a box plot of `distance_km`. Are there any outliers?